# Project 2: The three-body problem
*Solving a chaotic motion problem using deep neural networks*

**Learning from data [TIF285], Chalmers, Fall 2026**  
*Last modified: 2026-09-17*


<a id="toc"></a>
## Table of contents

- [Instructions](#instructions)
- [Project description](#description)
- [Physics background](#background)
- [Setup](#setup)
- [The data](#data)
- [Helper functions](#helpers)
- [The pre-trained network of Breen et al.](#pretrained)
- [List of tasks](#tasks)
- [Extra task: A Bayesian neural network](#extra)


<a id="instructions"></a>
## Instructions

This project has two hand-ins on Canvas, but only the notebook is required
from every group:

- **Your solution notebook (required).** Hand in this notebook, completed, via
  the Canvas assignment for Project 2. Upload also your trained network,
  saved as `myNN.pt` with
  `torch.save(model.state_dict(), 'myNN.pt')`.
- **A written report (for one project of your choice).** You write *one*
  report during the course, on either Project 1 or Project 2, and you choose
  which. It has its own Canvas assignment and its own deadline. If you write it
  on Project 1, you still hand in the solution notebooks for both projects.

The deadlines, the examination rules, and the requirements on the report are
given on the course web page (*Projects*) and on the corresponding Canvas
assignments. Read them before you start.

- The project is performed in groups of two students. Join a group in Canvas
  before submitting, and note that you must not work with the same partner on
  both projects.
- The extra task at the end is optional.
- Students are allowed to discuss and help each other, but every student must
  understand their submitted solution in the sense that they should be able to
  explain and discuss it with a peer or with a teacher.


#### What we expect to find in the notebook

Every task asks for something to look at, not only code to run:

| Task | Expected in the notebook |
|---|---|
| 1 | the number of usable trajectories, and a figure comparing one trajectory from the data with the prediction of the pre-trained network |
| 2 | the parameter counts of your network and of the pre-trained one, and the MAE-vs-epoch figure for the training and the validation data |
| 3 | trajectory figures for a few training and a few validation examples, each showing the data, the pre-trained network and your network |
| 4 | the measured time for one prediction, and the comparison with `Brutus` |
| 5 | the figure of the perturbed initial conditions, for both networks |
| Extra | the free energy during training, the predictive-band figure, continued past the end of the data, and the calibration table |

Before you hand in: **restart the kernel, run all cells, and save the notebook
with its outputs.** We grade what is in the file, so a notebook whose output
cells are empty counts as incomplete. Add a sentence of interpretation next to
each figure. A figure with no comment is hard to give credit for.

Training the network takes some minutes. See the comments and hints below if
you want to avoid re-training every time you run the notebook from the start.

<div style="text-align: right"><a href="#toc">&#8593; Table of contents</a></div>


<a id="description"></a>
## Project description

### Learning goals
- Apply a deep neural network as an emulator for a physics problem whose direct
  numerical solution is difficult and expensive.
- Get familiar with [PyTorch](https://pytorch.org/), a state-of-the-art machine-learning
  library.
- Get experience with the diagnostics used when training a neural network, and
  with the choices and difficulties involved.
- Put together ideas and tools from several parts of the course.
- Reproduce scientific results from a published paper.

### Main task
The overall goal is to reproduce several of the results in
[*Newton vs the machine: solving the chaotic three-body problem using deep
neural networks*](https://arxiv.org/abs/1910.07291) by Philip G. Breen,
Christopher N. Foley, Tjarda Boekholt and Simon Portegies Zwart.

The authors have graciously agreed that we use the raw data from their study for
this project. Please do not distribute it further.

<div style="text-align: right"><a href="#toc">&#8593; Table of contents</a></div>


<a id="background"></a>
## Physics background

#### The three-body problem
We study the motion of three bodies of equal mass moving under their mutual
classical gravitational attraction. Because the resulting motion is chaotic, the solution
for a given initialization can only be found by iterative numerical
computation, at a potentially very large computational cost.

Here you will demonstrate that a deep artificial neural network (ANN) can learn
the solutions of the equations of motion from a set of numerical training data.
The trained network then provides good approximations for new initial conditions
at a predictable, and small, computational cost.

The training data was generated with the state-of-the-art numerical code
`Brutus`
[[Boekholt, T., Portegies Zwart, S. On the reliability of N-body simulations.
Comput. Astrophys. 2, 2 (2015)](https://doi.org/10.1186/s40668-014-0005-3)].
You will not run this code, as the data is already generated. However, its 
run time for the trajectories studied here ranges from minutes up to hours,
so it will be relevant to ask at the end what a prediction with the network
costs instead.

#### Initial condition and choice of coordinates
- We work in the centre-of-mass (CM) frame and consider only the scenario in
  which the three masses start from rest.
- It can be shown, without loss of generality, that the coordinate system can
  always be chosen so that the three particles $p_1, p_2, p_3$ move in a plane,
  so that only $(x, y)$ coordinates are needed.
- The $(x, y)$ coordinates can furthermore be chosen and scaled such that
  particle 1 starts at $(1, 0)$ and particle 2 starts in the second quadrant
  ($x < 0$, $y > 0$).
- Since we are in the CM frame and the three masses are equal, we always have
  $x_1 + x_2 + x_3 = 0$ and $y_1 + y_2 + y_3 = 0$. The position of particle 3
  can therefore be determined from the positions of particles 1 and 2.

See also Fig. 1 of the paper.

The units are chosen such that $G = 1$ and all three masses are $m = 1$.

<div style="text-align: right"><a href="#toc">&#8593; Table of contents</a></div>


<a id="setup"></a>
## Setup

All imports used in this notebook are collected here. Add your own below if you
need more.


In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

### SET YOUR OWN SEED HERE ###
# Set your own seed here, so that your results are reproducible -- and your own
# rather than every other group's. Any integer will do; your Canvas group
# number is a natural choice. State it in your report.
SEED = 2026                      # <-- change this
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
### END SEED SETTING ###

# Training a small network on a few million samples is quite feasible on a CPU.
# If you happen to have a GPU (`cuda`) or an Apple-silicon GPU (`mps`), it will
# be used instead.
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'PyTorch {torch.__version__}, running on {device}')

<a id="data"></a>
## The data

The data is made available as a compressed numpy file on the Canvas course page.
Because of its size (about 500 MB) it is *not* included in the course git
repository. Navigate to *Files > Project2* and download `data_project2.npz` into
the directory that holds this notebook.

- The first axis labels the 9000 data samples, i.e. the trajectories.
- The second axis is the 1000 time steps, corresponding to 3.9 time units. The
  original data of the paper has 2561 time steps, reaching 10 time units, but as
  discussed there the training works better on the shorter interval, so that is
  what has been kept here.
- The nine columns of the last axis are
  $[t, x_1, y_1, x_2, y_2, v_{x,1}, v_{y,1}, v_{x,2}, v_{y,2}]$. Recall that the
  position of particle 3 follows from the positions of particles 1 and 2. The
  velocity columns are **not used** in any task.
- **Some trajectories contain a collision**, after which the positions of the
  particles are stuck at $(0, 0)$. These must be found and removed before you
  train on the data; that is the first thing asked for in the list of tasks.


In [ ]:
# Load and unpack the compressed numpy array.
load_data = np.load('data_project2.npz')
data = load_data['arr_0']

print(f'data.shape = {data.shape}   '
      '(trajectories, time steps, [t, x1, y1, x2, y2, vx1, vy1, vx2, vy2])')

tlist = data[0][:, 0]
print(f'{len(tlist)} time steps in the range '
      f'[{tlist[0]:.2f}, {tlist[-1]:.2f}] time units')

<a id="helpers"></a>
## Helper functions

These small helper functions are suggested, but you need to write the code. You are allowed to construct a solution using your own helper functions.


In [ ]:
def get_trajectories(pred):
    """
    Get the positions of all three particles from a network output.

    Args:
        pred (array): An array of shape (N, 4), where N is the number of time
                      steps. The four columns give the positions of particles
                      1 and 2 at every time step.

    Returns:
        p1, p2, p3 (tuple of arrays): Three arrays of shape (N, 2) giving the
                      (x, y) positions of the three particles.
    """
##################
# YOUR CODE HERE #
##################


def plot_trajectories(p1, p2, p3, ax=None, label=None, **kwargs):
    """
    Plot the trajectories of the three particles, marking the starting points.

    Args:
        p1, p2, p3 (array): Three arrays of shape (N, 2), the (x, y) positions.
        ax (axis object): Default None, in which case a new one is created.
        label (str): Legend label. It is attached to the first curve only, so
                     that one call adds one legend entry rather than three.
        kwargs (dict): Further keyword arguments passed on to `ax.plot`.

    Returns:
        ax: The axes object.
    """
##################
# YOUR CODE HERE #
##################

def predict(net, x):
    """
    Evaluate a network on numpy input and return a numpy array.

    Puts the network in evaluation mode and switches off gradient tracking,
    which is what you want whenever you are not training.

    Args:
        net (nn.Module): The network.
        x (array): An array of shape (N, 3), the network input.

    Returns:
        (array): An array of shape (N, 4), the predicted positions.
    """
##################
# YOUR CODE HERE #
##################


In [ ]:
def get_data(idx):
    """
    Get one training instance from the data set, at index idx.

    The data is assumed to be in the array `data`.

    Args:
        idx (int): Which trajectory to fetch.

    Returns:
        x (array): An array of shape (time_steps, 3), the input to the network.
                   The first column is the time and the second and third columns
                   are the (x, y) coordinates of the *initial* position of the
                   second particle, repeated for every time step. The first
                   particle always starts at (1, 0) and the third particle
                   follows from the first two.

        y (array): An array of shape (time_steps, 4), the output expected from
                   the network. The first two columns are the (x, y) coordinates
                   of the first particle and the last two those of the second
                   particle, at the time given in the first column of x.
    """
##################
# YOUR CODE HERE #
##################


<a id="pretrained"></a>
## The pre-trained network of Breen et al.

The network trained by the authors of the paper is distributed with this
notebook as `Breen_NN_project2.pt`, a PyTorch *state dictionary*: the learned
weights and biases, but not the architecture. To use it you must first build a
network of exactly the right topology and then load the weights into it. That
topology is given below: ten hidden layers of width 128 with `relu`
activations, three inputs and four outputs.

The **input** to the network is $[t, x_2, y_2]$: the time at which the positions
should be predicted, and the *initial* position of particle 2. Particle 1 always
starts at $(1, 0)$, so no further input is needed. The **output** is
$[x_1, y_1, x_2, y_2]$, the positions of particles 1 and 2 at that time.

A whole trajectory is therefore predicted by one call on an input array of shape
$1000 \times 3$, whose first column runs over the time steps while the second
and third columns simply repeat the initial position.


In [ ]:
def breen_topology(depth=10, width=128):
    """Build an empty network with the architecture used by Breen et al."""
    layers, n_in = [], 3
    for _ in range(depth):
        layers += [nn.Linear(n_in, width), nn.ReLU()]
        n_in = width
    layers += [nn.Linear(n_in, 4)]
    return nn.Sequential(*layers)


pre_trained_model = breen_topology().to(device)
pre_trained_model.load_state_dict(torch.load('Breen_NN_project2.pt'))
pre_trained_model.eval()

print(pre_trained_model)

<div style="text-align: right"><a href="#toc">&#8593; Table of contents</a></div>

<a id="tasks"></a>
## List of tasks

1. **Get the data and the network of Breen et al. working.** Some of the
   trajectories contain a collision and have to be found and removed. Write a
   function that turns a trajectory into the input/output that the network
   works with, and satisfy yourself that you have it right before you train
   anything of your own.
1. **Build and train your own network**, and compare its number of trainable
   parameters with theirs. Show the mean absolute error against epoch for both
   the training and the validation data, as in Fig. 3 of the paper.
1. **Compare trajectories.** Plot the data, the pre-trained network and your own
   network together for several trajectories (from both the training set and
   the validation set), as in Fig. 4, and comment on how
   yours performs. You should not expect it to match theirs. Note also that the
   trajectories shown in the paper appear to be ones the model handles
   particularly well, so choose yours honestly.
1. **Measure what a prediction costs**, and compare with `Brutus`, which
   according to the paper takes minutes to hours for a single trajectory.
1. **Explore the chaos.** Generate a hundred trajectories from slightly
   disturbed initial conditions and reproduce Fig. 5 of the paper, with both
   networks. Describe what happens to the hundred points as time increases, and
   say in what sense the emulator is still useful once they have spread out.

<div style="text-align: right"><a href="#toc">&#8593; Table of contents</a></div>


#### Suggestions for how to proceed
- Follow the lead of the PyTorch demonstrations of the course (secs. 27.3 and
  27.4).
- The helper functions above are a suggested decomposition. You are free to
  structure your solution differently.
- **Design your own network rather than copying the paper's.** The depth and the
  width, the optimizer and its settings, the batch size and the number of epochs
  are all yours to choose; say what you chose and why. If you would rather not
  begin from nothing, a starting point that is known to work is five hidden
  layers of width 64 with `relu` activations,
  `torch.optim.Adam(model.parameters(), lr=0.001, betas=(0.5, 0.5))`,
  `nn.L1Loss()`, batches of 5000 rows and of order 100 epochs. The paper used
  ten hidden layers of width 128, which costs considerably more to train.
- Use the mean absolute error as the loss, so that your learning curve can be
  read against Fig. 3 of the paper.
- Hold out some of the data for validation; a tenth is a reasonable choice. This
  much is *not* a free choice: make the split **between whole trajectories**.
  Two rows from one trajectory share an initial condition and differ only in
  $t$, so splitting row by row would leave the validation set holding
  near-duplicates of rows the network trained on.
- You are not expected to tune systematically --- training on the whole data set
  is expensive enough that a search is out of scope --- but you are expected to
  say what you settled on.


#### Comments and hints
- **The input and the output.** The input is $[t, x_2, y_2]$: the time at which
  the positions should be predicted, and the *initial* position of particle 2.
  Particle 1 always starts at $(1, 0)$, so nothing further is needed. The output
  is $[x_1, y_1, x_2, y_2]$ at that time. A whole trajectory is a single call on
  an array of shape $1000 \times 3$, whose first column runs over the time
  steps while the second and third simply repeat the initial position.
- **Check the pipeline before you train anything.** Predict one trajectory with
  the pre-trained network and plot it on top of the data. Putting the
  *time-dependent* position of particle 2 into the input instead of the initial
  one is the classic mistake, it makes the input contain the answer, and it is
  invisible until you draw the picture.
- A collision leaves all four position columns at exactly zero for the rest of
  the trajectory, so testing the last time step is enough. About one trajectory
  in six is affected.
- Use `float32`. It is PyTorch's default precision, and mixing it with
  `float64` gives a type error the first time you call the network.
- Shuffle the rows freely inside the training loop. Every row is an independent
  (input, target) pair, and a batch drawn at random from the whole training set
  trains distinctly better than one covering five consecutive trajectories.
  Shuffle the *indices*, with `torch.randperm`, rather than the arrays, so that
  you can still pick out individual trajectories afterwards.
- Measure the training and the validation error the same way, with the weights
  the epoch ended on. A running average over the epoch lags behind, because the
  model improves while the average is being accumulated, and the two curves can
  then not be compared with each other.
- Do not show only the trajectories that look good. Ranking a sample of them by
  the error of the pre-trained network and picking a spread of percentiles is a
  tidy way of keeping the choice honest.
- `%timeit` reports the mean and the standard deviation over many runs. Time a
  whole trajectory rather than a single time step: the network is evaluated on
  the batch of 1000 inputs in one call, and that is what makes it fast.
- Training every time you run the notebook is tedious. You can save the trained
  network with `torch.save(model.state_dict(), 'myNN.pt')` --- which is in any
  case one of the things you hand in --- and load it back with
  `model.load_state_dict(torch.load('myNN.pt'))`. Remember that the training
  history is not stored in the state dictionary, and that the version you hand
  in has to run from a restarted kernel.

<div style="text-align: right"><a href="#toc">&#8593; Table of contents</a></div>


<a id="extra"></a>
## Extra task: A Bayesian neural network

Everything so far has given point predictions. Comparing trajectories showed
that their quality varies enormously. Between the 10th and the 97th percentile of difficulty the
error of the pre-trained network changes by a factor of forty. Nothing in
the output says which case you are looking at. An emulator used inside an
inference, which is what emulators are for, has to carry an error bar, or the
inference is wrong in a way nobody can see.

Here you will implement an approach to quantify error bars for neural network
predictions. This will be achieved by assigning the weights of the network a
distribution instead of a single value. Sampling that weight posterior directly
is not an option here: your network has of order $10^4$ parameters, and ensemble
samplers such as `emcee` degrade badly above a few tens of dimensions. The
tractable alternative is variational inference, and the background for it is in
the lecture notes: *Bayesian neural networks in practice* for variational
inference, the variational free energy and Bayes by Backprop, and the
demonstration *Bayes by Backprop in PyTorch* for the implementation. Read those
first; this task assumes them.

*BNN uncertainties.* In the BNN demonstration from the book, the predictive spread was split
into an **epistemic** part, from uncertainty about the weights, and an
**aleatoric** part, from noise in the data. In this project there is no data noise: `Brutus`
returns exactly one trajectory for a given initial condition, and the data are
deterministic. However, there are still two types of uncertainty:
the kind that more data could in principle reduce, and the kind that it cannot.

*Where the uncertainties enter.*
The epistemic uncertainty is reflected in the probability distribution for the weights $\boldsymbol{w}$.
To incorporate the aleatoric one, you will construct the Bayesian network with **eight** outputs to predict the four positions:
four means and four (log-)variances. For a given input $\boldsymbol{x}$ and a
given draw of the weights $\boldsymbol{w}$, one forward pass produces both
$\boldsymbol{\mu_w}(\boldsymbol{x})$ and $\boldsymbol{\sigma_w}(\boldsymbol{x})$,
and the model for the data is
$\boldsymbol{y} \sim \mathcal{N}(\boldsymbol{\mu_w}(\boldsymbol{x}),
\boldsymbol{\sigma_w}(\boldsymbol{x})^2)$. The width is not a separate parameter
with a posterior of its own: it is computed from $\boldsymbol{x}$ by the same
weights as the mean, and it changes when $\boldsymbol{w}$ is redrawn exactly as
the mean does. That is why the recipe in the hints averages the *variances* over
weight draws.

*What determines the uncertainties.* What you optimise is
$\boldsymbol{\theta} = (\boldsymbol{\mu}, \boldsymbol{\rho})$, by minimising the
variational free energy

$$
\mathcal{F}/N = \mathbb{E}_q\left[-\log
p(\boldsymbol{y}|\boldsymbol{x},\boldsymbol{w})\right]
+ D_{\mathrm{KL}}\left[q(\boldsymbol{w}|\boldsymbol{\theta}) \,\|\,
p(\boldsymbol{w})\right] \big/ N .
$$

Both terms depend on $\boldsymbol{\theta}$, but only the first involves the
data. The second measures how far the variational distribution has been dragged
from the prior, and is a function of the weight distribution alone.

So the data input that shapes $\boldsymbol{\sigma_w}(\boldsymbol{x})$ comes
entirely from the likelihood term: for a Gaussian likelihood this term is smallest when
$\sigma^2(\boldsymbol{x})$ matches the typical squared residual near
$\boldsymbol{x}$. The KL term does not push the
width anywhere in particular (it never sees $\boldsymbol{y}$) but it does
regularise the weights that produce the width, and so limits how sharply
$\boldsymbol{\sigma_w}$ can vary.

Here there is no noise in the training data,
so what $\boldsymbol{\sigma_w}(\boldsymbol{x})$ converges to
instead is a measure of **the network's own error**: large where it cannot
follow the motion, small where it can. That is a *discrepancy* term, a statistical model of a
systematic error, not a description of noise.

The epistemic and the discrepancy terms answer different questions: how uncertain the fit still is, against how badly the model fails once the fit has settled. Reporting only their sum hides which of the two you are up against. Only the first is reducible by training on more trajectories, and one of the things you will find in step 3 is how little of the total it accounts for.

### What to do
1. **Implement a Bayesian version of your network and train it**, with the
   eight-output head described above. Train on a few hundred trajectories, not
   the whole set. That number is the largest of the three budgets you will use in step 3.
   Show that the training converged: report the free energy, and the KL term
   separately, as the fit proceeds.
1. **Plot the predictive band** for the easiest and the hardest validation
   trajectory, showing the epistemic and the total width separately. Does the
   band widen where the prediction goes wrong? Continue the prediction past the
   end of the training data --- the trajectories stop at $t = 3.9$, but nothing
   prevents you from asking the network for later times --- and mark where the
   data end. What does each of the two widths do out there, and which of them
   would you trust as a warning?
1. **Check the calibration.** What fraction of held-out points falls inside the
   68% and the 95% band? Then repeat with two substantially smaller training
   sets and tabulate all three. (The reference solution uses 500, 100 and 20
   trajectories.) A predictive distribution that is not calibrated is not
   useful, however Bayesian its derivation.
1. **A question to close on, if you write your report on this project.** The
   last of the main tasks showed the spread produced by perturbing the initial
   condition. This task
   gives a second, independent spread produced by the emulator's own ignorance.
   Which dominates, and at what time? What would it mean for a scientific
   conclusion if you reported one and not the other?

<div style="text-align: right"><a href="#toc">&#8593; Table of contents</a></div>


#### Suggestions how to proceed and what to consider
- The structure for building a Bayesian layer, the analytic KL and the training loop are all in the
  demonstration notebook.
- Per data point the cost is
  $\mathcal{F}/N = \mathbb{E}_q[-\log p(y|x,w)] + D_{\mathrm{KL}}/N$, so the KL
  term must be divided by the size of the **training set**, not the batch.
- With a predicted log-variance the loss is the Gaussian negative log
  likelihood, $\tfrac12 \sum_k [\log v_k + (y_k - \mu_k)^2 / v_k]$ up to a
  constant. Restrict the predicted log-variance to a fixed range (`.clamp(-12, 4)`
  in PyTorch, i.e. $\sigma$ between roughly 0.003 and 7). Without a floor on the
  variance, one bad batch drives it towards zero, the loss overflows, and every
  subsequent value is a NaN.
- The variational width is $\sigma_q = \mathrm{softplus}(\rho)$, so it is $\rho$,
  not $\sigma_q$ itself, that the optimiser adjusts. Initialise it small and
  negative (e.g. $\rho_0 = -5$, giving $\sigma_q \approx 0.007$), so that the
  network starts almost deterministic and the data widen the posterior from
  there. Starting wide fails quietly rather than loudly: the run completes and
  the figures look plausible, but the epistemic term stops responding to where
  the data are.
- For the prior, $\boldsymbol{w} \sim \mathcal{N}(0, 2/n_\mathrm{in})$ per layer
  is a good choice: it is the `relu` initialisation distribution, and a
  $1/n_\mathrm{in}$ scaling of this kind gives the Gaussian process of the
  wide-network limit.
- A workable budget is of order 20 000 gradient steps at batch 2000, a few
  minutes per training. As in the main tasks, this is a starting point rather
  than a requirement. Reuse the `x_train`, `y_train`, `x_val`, `y_val` you built
  there and slice off the first $N \times 1000$ rows, since those are already
  grouped by trajectory.
- *Uncertainty quantification.* Draw $S$ weight samples
  $\boldsymbol{w}_s \sim q$; each gives a mean $\mu_s(x)$ and a variance
  $\sigma_s^2(x)$ for every output. From those same $S$ forward passes: the
  predictive **mean** is the average of $\mu_s$ over $s$ and the **epistemic**
  variance is the *variance of* $\mu_s$ over $s$, i.e. how much the mean moves
  as the weights are redrawn; the **discrepancy** variance is the *average of*
  $\sigma_s^2$ over $s$; and the total variance is the sum of the two. A few
  tens of draws is enough.